# 06 — Train EBM Verifier Head

Trains a tiny linear scoring head (`VerifierHead`) on top of the **frozen** Qwen3-4B backbone
already loaded in the notebook. The head learns to rank correct solutions above incorrect ones
using a **Bradley-Terry ranking loss**.

**Why this works without a second model:**
- The backbone is the same `llm` used for generation — reused in *scoring mode* via `llm.forward(output_hidden_states=True)`
- The head is `nn.Linear(2560, 1)` — only **2,561 trainable parameters** (~10 KB)
- Zero additional VRAM for weights; scoring is a single matrix multiply through all 36 layers

**Run order:**
1. `02_inference.ipynb` — full public inference (produces `adaptive_public_v2_results.jsonl`)
2. `03_qlora_finetune.ipynb` — optional SFT
3. `04_grpo_train.ipynb` — optional GRPO
4. **This notebook** — train EBM head (~20-30 min)
5. `02_inference.ipynb` re-run (or `05_private_submission.ipynb`) with EBM active

## 1. Config

In [ ]:
import json, os, re, sys, random
from pathlib import Path


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))

# ── Competition constraint: ONLY this model ───────────────────────────────────
# Verifier backbone = same model that does generation (llm reused, zero extra VRAM).
MODEL_ID     = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID       = "0"

PREV_RESULTS = REPO_ROOT / "artifacts" / "logs" / "runs" / "adaptive_public_v2_results.jsonl"
PUBLIC_PATH  = REPO_ROOT / "data" / "raw" / "public.jsonl"
EBM_OUTPUT   = REPO_ROOT / "artifacts" / "models" / "ebm_verifier_v1"

# ── Training hyperparams ─────────────────────────────────────────────────────
# Higher LR is fine for a single linear layer (no LoRA needed — only 2,561 params).
LEARNING_RATE  = 1e-3
EPOCHS         = 10
BATCH_SIZE     = 8     # pairs per gradient step
GRAD_ACCUM     = 2     # effective batch = 16
MAX_SEQ_LENGTH = 4096  # question + solution tokens
VAL_FRACTION   = 0.15  # held-out pairs for ranking-accuracy check
RANDOM_SEED    = 42

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
    DRIVE_BASE   = Path('/content/drive/MyDrive/math-qa-llm')
    PREV_RESULTS = DRIVE_BASE / "artifacts" / "logs" / "runs" / "adaptive_public_v2_results.jsonl"
    PUBLIC_PATH  = DRIVE_BASE / "data" / "raw" / "public.jsonl"
    EBM_OUTPUT   = DRIVE_BASE / "artifacts" / "models" / "ebm_verifier_v1"
except ImportError:
    IS_COLAB = False

assert PREV_RESULTS.is_file(), (
    f"Results file not found: {PREV_RESULTS}\n"
    "Run 02_inference.ipynb with N_QUESTIONS=None first."
)
assert PUBLIC_PATH.is_file(), f"public.jsonl not found: {PUBLIC_PATH}"

print(f"REPO_ROOT    : {REPO_ROOT}")
print(f"PREV_RESULTS : {PREV_RESULTS}  | exists: {PREV_RESULTS.is_file()}")
print(f"EBM_OUTPUT   : {EBM_OUTPUT}")
print(f"EPOCHS={EPOCHS}  LR={LEARNING_RATE}  BATCH={BATCH_SIZE}×ACCUM={GRAD_ACCUM}")

## 2. Build Training Pairs

Three tiers of training signal extracted from the inference results:

| Source | Positive (correct) | Negative (wrong) | Quality |
|--------|--------------------|------------------|---------|
| Phase 2 flipped questions | Phase 2 response (correct) | `phase1_response` (wrong) | ★★★ Best — same model, same question, natural contrast |
| Wrong Phase 1 questions | Gold answer string | Phase 1 response (wrong) | ★★ Good |
| Correct-only questions | Phase 1 response (correct) | Shuffled response from different question | ★ Synthetic |

In [ ]:
# ── Load public.jsonl ─────────────────────────────────────────────────────────
with open(PUBLIC_PATH, encoding="utf-8") as f:
    public_items = {str(item["id"]): item for item in (json.loads(l) for l in f)}

# ── Load inference results ────────────────────────────────────────────────────
results = {}
with open(PREV_RESULTS, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        results[str(rec["id"])] = rec

print(f"Public questions: {len(public_items)}")
print(f"Inference results: {len(results)}")

# ── Build pairs ───────────────────────────────────────────────────────────────
pairs = []
correct_responses = []   # collect for synthetic negatives later

for qid, item in public_items.items():
    res = results.get(qid)
    if not res:
        continue

    question = item["question"]
    options  = item.get("options")
    gold     = item.get("answer")

    if res.get("phase_used") == 2 and res.get("correct") and res.get("phase1_response"):
        # BEST: Phase 2 flipped — Phase 1 was wrong, Phase 2 was correct
        pairs.append({
            "question": question, "options": options,
            "positive": res["response"],         # Phase 2 correct
            "negative": res["phase1_response"],   # Phase 1 wrong
            "source":   "phase_flip",
        })
    elif not res.get("correct"):
        # GOOD: model was wrong — gold answer text vs. model's wrong response
        if gold is not None:
            gold_str = (
                ", ".join(str(g) for g in gold) if isinstance(gold, list) else str(gold)
            )
            gold_boxed = f"\\boxed{{{gold_str}}}"
            pairs.append({
                "question": question, "options": options,
                "positive": gold_boxed,      # gold answer
                "negative": res["response"], # wrong model attempt
                "source":   "gold_vs_wrong",
            })
    else:
        # Correct Phase 1 — save for synthetic negative construction
        correct_responses.append({
            "question": question, "options": options,
            "response": res["response"],
        })

# ── Synthetic negatives: pair each correct response with a different question's correct response
rng = random.Random(RANDOM_SEED)
if len(correct_responses) >= 2:
    shuffled = correct_responses[:]
    rng.shuffle(shuffled)
    # Pair original[i] with shuffled[i] — guaranteed different (or re-shuffle until it is)
    for orig, neg_src in zip(correct_responses, shuffled):
        if orig["response"] == neg_src["response"]:
            continue   # skip exact same response (rare)
        pairs.append({
            "question": orig["question"], "options": orig["options"],
            "positive": orig["response"],
            "negative": neg_src["response"],   # correct answer to a DIFFERENT question
            "source":   "synthetic",
        })

source_counts = {}
for p in pairs:
    source_counts[p["source"]] = source_counts.get(p["source"], 0) + 1

print(f"\nTotal training pairs: {len(pairs)}")
for src, cnt in sorted(source_counts.items()):
    print(f"  {src}: {cnt}")

if len(pairs) < 50:
    print("\nWARNING: fewer than 50 pairs — run 02_inference.ipynb with N_QUESTIONS=None first")

## 3. Load Model (backbone for scoring)

The same `Qwen3-4B-Thinking-2507` used for generation is reused here as the verifier backbone.
All its parameters are **frozen** — only the `VerifierHead` linear layer trains.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"   # right-pad for classification (last-token pooling)

_vram_gb  = (torch.cuda.get_device_properties(0).total_memory / 1e9
             if torch.cuda.is_available() else 0)
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"GPU: {_gpu_name} | VRAM: {_vram_gb:.1f} GB")

if _vram_gb >= 20:
    _mkw    = dict(dtype=torch.bfloat16, device_map="auto",
                   trust_remote_code=True, attn_implementation="sdpa")
    _qlabel = "BF16+SDPA"
else:
    from transformers import BitsAndBytesConfig
    _mkw    = dict(
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        ),
        device_map="auto", trust_remote_code=True, attn_implementation="sdpa",
    )
    _qlabel = "NF4+SDPA"

llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_mkw)
llm.eval()

# Freeze ALL backbone parameters — only the linear head will train
for p in llm.parameters():
    p.requires_grad_(False)

print(f"Model loaded ({_qlabel}) | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Backbone params: {sum(p.numel() for p in llm.parameters()):,} (all frozen)")

## 4. Build VerifierHead + Tokenization

**Input format** (both training and inference):
```
<|im_start|>system
You are evaluating whether this mathematical solution is correct and complete.
<|im_end|>
<|im_start|>user
{question}\n\nOptions:\n... (if MCQ)
<|im_end|>
<|im_start|>assistant
{candidate_solution}
<|im_end|>
```
The last-token hidden state of this sequence encodes the full `(question, solution)` context.

In [ ]:
class VerifierHead(nn.Module):
    """Tiny linear scoring head on top of frozen Qwen3-4B backbone.

    Last-token pooling: takes the hidden state of the last real token in the
    (question + solution) sequence and projects it to a scalar score.
    Higher score = the backbone 'thinks' this is a more correct solution.

    IMPORTANT: we call `base_model.model` (the inner Qwen3Model), NOT `base_model`
    (which is Qwen3ForCausalLM). This skips the lm_head (150K-vocab projection)
    that would otherwise allocate ~10 GB of logits per forward pass and OOM the
    A30. The EBM only needs hidden states — logits are irrelevant.
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.head = nn.Linear(hidden_size, 1, bias=True)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def score(self, base_model, input_ids, attention_mask) -> torch.Tensor:
        """(batch,) scalar scores. base_model parameters receive no gradients."""
        # Use the inner backbone — bypasses lm_head, returns last_hidden_state directly.
        backbone = base_model.model if hasattr(base_model, "model") else base_model
        with torch.no_grad():
            hidden = backbone(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            ).last_hidden_state                          # (batch, seq_len, hidden)
        seq_lengths       = attention_mask.sum(dim=1) - 1   # (batch,) last real token idx
        last_token_hidden = hidden[
            torch.arange(hidden.size(0), device=hidden.device),
            seq_lengths,
        ]                                                    # (batch, hidden)
        return self.head(last_token_hidden).squeeze(-1)      # (batch,)


def _get_last_token_hidden(enc: dict) -> torch.Tensor:
    """Run llm backbone forward pass and return last-token hidden state. Detached.

    Calls `llm.model` (Qwen3Model) — NOT `llm` (Qwen3ForCausalLM) — to avoid
    computing the LM-head logits, which would OOM at batch×seq=32K.
    """
    backbone = llm.model if hasattr(llm, "model") else llm
    with torch.no_grad():
        hidden = backbone(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            use_cache=False,
        ).last_hidden_state                              # (batch, seq_len, hidden)
    seq_lengths = enc["attention_mask"].sum(dim=1) - 1   # (batch,)
    return hidden[
        torch.arange(hidden.size(0), device=hidden.device),
        seq_lengths,
    ]                                                    # (batch, hidden) — detached


hidden_size   = llm.config.hidden_size   # 2560 for Qwen3-4B
verifier_head = VerifierHead(hidden_size).to(llm.device).to(torch.bfloat16)

n_params = sum(p.numel() for p in verifier_head.parameters())
print(f"VerifierHead hidden_size={hidden_size} | trainable params: {n_params:,}")
# Expected: 2560 * 1 + 1 = 2,561


_VERIFIER_SYSTEM = "You are evaluating whether this mathematical solution is correct and complete."


def format_for_verifier(question: str, options, candidate: str) -> str:
    """Build the input string the verifier scores. Same format used at inference time."""
    user = question
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        user  += "\n\nOptions:\n" + "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))
    msgs = [
        {"role": "system",    "content": _VERIFIER_SYSTEM},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": candidate},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)


def tokenize_texts(texts: list) -> dict:
    """Right-pad tokenized batch for scoring (last-token pooling needs real last token at end)."""
    enc = tokenizer(
        texts, padding=True, truncation=True,
        max_length=MAX_SEQ_LENGTH, return_tensors="pt",
    )
    return {k: v.to(llm.device) for k, v in enc.items()}

## 5. Train/Val Split

In [ ]:
rng = random.Random(RANDOM_SEED)
all_pairs = pairs[:]
rng.shuffle(all_pairs)

n_val    = max(10, int(len(all_pairs) * VAL_FRACTION))
val_pairs  = all_pairs[:n_val]
train_pairs = all_pairs[n_val:]

print(f"Train pairs: {len(train_pairs)}")
print(f"Val   pairs: {len(val_pairs)}")

## 6. Bradley-Terry Training Loop

**Loss:** `BT(pos, neg) = -log(sigmoid(score_pos - score_neg))`

This directly maximizes the probability that the model assigns a higher score to the correct solution. Only the `VerifierHead` linear layer accumulates gradients — the backbone is frozen.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


def bt_loss(pos_logits: torch.Tensor, neg_logits: torch.Tensor) -> torch.Tensor:
    """Bradley-Terry ranking loss. Maximizes P(correct > wrong)."""
    return -F.logsigmoid(pos_logits - neg_logits).mean()


optimizer = AdamW(verifier_head.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = (len(train_pairs) * EPOCHS) // (BATCH_SIZE * GRAD_ACCUM)
scheduler   = CosineAnnealingLR(optimizer, T_max=max(1, total_steps))

verifier_head.train()
step = 0

for epoch in range(EPOCHS):
    rng.shuffle(train_pairs)
    epoch_loss = 0.0
    n_batches  = 0

    optimizer.zero_grad()

    for i in range(0, len(train_pairs), BATCH_SIZE):
        batch = train_pairs[i : i + BATCH_SIZE]
        if not batch:
            continue

        pos_texts = [format_for_verifier(p["question"], p["options"], p["positive"]) for p in batch]
        neg_texts = [format_for_verifier(p["question"], p["options"], p["negative"]) for p in batch]

        pos_enc = tokenize_texts(pos_texts)
        neg_enc = tokenize_texts(neg_texts)

        # verifier_head.score() calls llm.forward() under torch.no_grad() internally
        # but the *head* itself is differentiable (its input = last_token_hidden.detach())
        pos_hidden = _get_last_token_hidden(pos_enc)
        neg_hidden = _get_last_token_hidden(neg_enc)

        pos_score = verifier_head.head(pos_hidden).squeeze(-1)
        neg_score = verifier_head.head(neg_hidden).squeeze(-1)

        loss = bt_loss(pos_score, neg_score) / GRAD_ACCUM
        loss.backward()

        epoch_loss += loss.item() * GRAD_ACCUM
        n_batches  += 1
        step       += 1

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(verifier_head.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    avg_loss = epoch_loss / max(1, n_batches)
    print(f"Epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")

# Flush any remaining gradient accumulation
if step % GRAD_ACCUM != 0:
    torch.nn.utils.clip_grad_norm_(verifier_head.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad()

verifier_head.eval()
print("Training complete.")

## 7. Validation — Ranking Accuracy

Checks what fraction of val pairs the verifier correctly ranks (positive score > negative score).

**Thresholds:**
- `< 55%` → training didn't converge; check pair quality or run more epochs
- `55–65%` → marginal; EBM will still help but weakly
- `> 65%` → good — save and proceed
- `> 75%` → excellent

In [ ]:
verifier_head.eval()
correct_rank = 0

with torch.no_grad():
    for pair in val_pairs:
        pos_text = format_for_verifier(pair["question"], pair["options"], pair["positive"])
        neg_text = format_for_verifier(pair["question"], pair["options"], pair["negative"])

        pos_enc = tokenize_texts([pos_text])
        neg_enc = tokenize_texts([neg_text])

        pos_h = _get_last_token_hidden(pos_enc)
        neg_h = _get_last_token_hidden(neg_enc)

        pos_score = verifier_head.head(pos_h).squeeze(-1).item()
        neg_score = verifier_head.head(neg_h).squeeze(-1).item()

        if pos_score > neg_score:
            correct_rank += 1

ranking_acc = correct_rank / len(val_pairs) * 100
print(f"Validation ranking accuracy: {correct_rank}/{len(val_pairs)} = {ranking_acc:.1f}%")

if ranking_acc < 55:
    print("WARNING: ranking accuracy < 55% — consider more epochs or checking pair quality")
elif ranking_acc >= 65:
    print("Good — EBM verifier is ready to save and use")

## 8. Save Verifier Head

Only the **head weights** (~10 KB) are saved. The backbone is the standard Qwen3-4B already on disk.

At inference time (`02_inference.ipynb` / `05_private_submission.ipynb`), the EBM load cell does:
```python
verifier_head = VerifierHead(llm.config.hidden_size).to(llm.device).to(torch.bfloat16)
verifier_head.load_state_dict(torch.load(EBM_PATH / "verifier_head.pt", map_location=llm.device))
verifier_head.eval()
```

In [ ]:
EBM_OUTPUT.mkdir(parents=True, exist_ok=True)

head_path = EBM_OUTPUT / "verifier_head.pt"
cfg_path  = EBM_OUTPUT / "verifier_config.json"

torch.save(verifier_head.state_dict(), head_path)

cfg_path.write_text(json.dumps({
    "backbone":        "Qwen/Qwen3-4B-Thinking-2507",
    "hidden_size":     hidden_size,
    "type":            "ebm_bradley_terry_v1",
    "trainable_params": n_params,
    "epochs":          EPOCHS,
    "learning_rate":   LEARNING_RATE,
    "val_ranking_acc": round(ranking_acc, 2),
    "n_train_pairs":   len(train_pairs),
    "n_val_pairs":     len(val_pairs),
}, indent=2))

head_kb = head_path.stat().st_size // 1024
print(f"Saved: {head_path}  ({head_kb} KB)")
print(f"Saved: {cfg_path}")
print()
print("Next steps:")
print("  1. Re-run 02_inference.ipynb (or 05_private_submission.ipynb)")
print("  2. The EBM load cell will detect verifier_head.pt automatically")
print("  3. PHASE2_N_SAMPLES will be bumped to 8, Phase 2 will use EBM reranking")